# P2P collectives

Implement collective patterns using only point-to-point tensor communication.

In [1]:
# ruff: noqa: F401, F841, I001

import flock
import flock.torch as ft
import torch
from dist_puzzles import check

## 1. Manual broadcast

Rank 0 sends the same tensor to every rank, including itself.

This is the simplest one-to-all pattern: a single source distributes identical data across the whole group. Think of it as rank 0 shouting a value that everyone, including itself, needs to hear.

In [2]:
value = torch.tensor([7])


@flock.distribute(workers=4)
async def manual_broadcast(value: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    if rank == 0:
        sends = [ft.isend(dst, value) for dst in range(world)]
        for work in sends:
            await work.wait()

    return await ft.recv(0)


expected = [torch.tensor([7])] * 4
check(manual_broadcast(value), expected)

## 2. Manual scatter

Rank 0 splits a tensor into one chunk per rank and sends each chunk to its owner.

Where broadcast gives everyone the same data, scatter gives each rank a distinct slice. Rank 0 holds the full tensor, partitions it, and distributes the pieces — useful for dividing a dataset or sharding a parameter across workers.

In [3]:
data = torch.arange(8)


@flock.distribute(workers=4)
async def manual_scatter(data: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    if rank == 0:
        x = data.reshape(world, 2)
        sends = [ft.isend(dst, x[dst]) for dst in range(world)]
        for work in sends:
            await work.wait()

    return await ft.recv(0)


check(
    manual_scatter(data),
    [torch.tensor([0, 1]), torch.tensor([2, 3]), torch.tensor([4, 5]), torch.tensor([6, 7])],
)

## 3. Manual gather

Every rank sends its tensor to rank 0, which collects them in order.

The inverse of scatter. Each rank contributes one piece; rank 0 assembles the full picture. Only rank 0 ends up with the complete result — all other ranks return `None`. Notice how rank 0 must recv from each source in a fixed order to keep the result deterministic.

In [4]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def manual_gather(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    await ft.isend(0, x).wait()

    if rank != 0:
        return None

    gathered = []
    for src in range(world):
        gathered.append(await ft.recv(src))
    return gathered


check(
    manual_gather(flock.per_rank(values)),
    [[torch.tensor([0]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])], None, None, None],
)

## 4. Manual all-gather

Every rank gathers all tensors so that every rank ends up with the full ordered list.

All-gather = gather + broadcast, but only the data, not the computation. Rank 0 collects all contributions and fans them back out. Every rank gets an identical list `[chunk_0, chunk_1, ..., chunk_{N-1}]`. This is what happens when a model needs the full activation after it was sharded across workers.

In [5]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def manual_all_gather(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    await ft.isend(0, x).wait()

    if rank == 0:
        gathered = [await ft.recv(src) for src in range(world)]
        sends = []
        for dst in range(world):
            for value in gathered:
                sends.append(ft.isend(dst, value))
        for work in sends:
            await work.wait()

    return [await ft.recv(0) for _ in range(world)]


gathered = [torch.tensor([0]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])]
check(
    manual_all_gather(flock.per_rank(values)),
    [gathered, gathered, gathered, gathered],
)

## 5. Manual reduce

Every rank sends a tensor to rank 0, which accumulates them into a single sum.

Like gather, but instead of keeping chunks separate, rank 0 folds them with an operation (here, addition). Only rank 0 holds the result. This is the building block for gradient aggregation in data-parallel training — each worker computes a local gradient, rank 0 sums them.

In [6]:
values = [torch.tensor([rank + 1]) for rank in range(4)]


@flock.distribute(workers=4)
async def manual_reduce(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    await ft.isend(0, x).wait()

    if rank != 0:
        return None

    total = torch.tensor([0])
    for src in range(world):
        total = total + await ft.recv(src)
    return total


check(manual_reduce(flock.per_rank(values)), [torch.tensor([10]), None, None, None])

## 6. All-reduce via reduce + broadcast

Reduce to rank 0, then broadcast the result so every rank holds the global sum.

This is the naive all-reduce: two sequential phases, both bottlenecked through rank 0. Every rank ends up with the same value. It works but doesn't scale — rank 0 is a throughput bottleneck and bandwidth grows linearly with world size. Puzzles 7 and 8 fix that with ring algorithms.

In [7]:
values = [torch.tensor([rank + 1]) for rank in range(4)]


@flock.distribute(workers=4)
async def reduce_broadcast_all_reduce(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()

    await ft.isend(0, x).wait()

    if rank == 0:
        total = torch.tensor([0])
        for src in range(world):
            total = total + await ft.recv(src)
        sends = [ft.isend(dst, total) for dst in range(world)]
        for work in sends:
            await work.wait()

    return await ft.recv(0)


expected = [torch.tensor([10])] * 4
check(reduce_broadcast_all_reduce(flock.per_rank(values)), expected)

## 7. Ring all-gather

Each rank passes its chunk rightward around the ring until every rank has received every chunk.

No root bottleneck: all ranks send and receive simultaneously. After `N-1` steps each rank has seen every chunk. The key insight is that each rank forwards what it just received in the next step — the ring acts as a conveyor belt. Total data sent per rank: `(N-1)` messages of one chunk each.

In [8]:
values = [torch.tensor([rank]) for rank in range(4)]


@flock.distribute(workers=4)
async def ring_all_gather(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    left = (rank - 1) % world
    right = (rank + 1) % world

    chunks = {rank: x}
    current = chunks[rank]
    for step in range(world - 1):
        send_work = ft.isend(right, current)
        current = await ft.recv(left)
        await send_work.wait()
        src = (rank - step - 1) % world
        chunks[src] = current

    return [chunks[src] for src in range(world)]


gathered = [torch.tensor([0]), torch.tensor([1]), torch.tensor([2]), torch.tensor([3])]
check(
    ring_all_gather(flock.per_rank(values)),
    [gathered, gathered, gathered, gathered],
)

## 8. Ring all-reduce

Two ring passes: first reduce-scatter (each rank accumulates one fully-reduced chunk), then all-gather (distribute the reduced chunks back).

This is the algorithm behind NCCL's all-reduce. Both phases take `N-1` steps and every rank sends/receives at full bandwidth the whole time — no bottleneck. After phase 1 each rank owns one fully-summed slice; after phase 2 every rank holds the complete result. Total bandwidth per rank: `2 * (N-1)/N * tensor_size`, which approaches `2x` as N grows.

In [ ]:
values = [torch.arange(rank * 10, rank * 10 + 4) for rank in range(4)]


@flock.distribute(workers=4)
async def ring_all_reduce(x: torch.Tensor):
    rank = flock.get_rank()
    world = flock.get_world_size()
    left = (rank - 1) % world
    right = (rank + 1) % world

    chunks = [chunk.clone() for chunk in x.chunk(world)]

    for step in range(world - 1):
        send_idx = (rank - step) % world
        recv_idx = (rank - step - 1) % world
        send_work = ft.isend(right, chunks[send_idx], tag=step)
        chunks[recv_idx] = chunks[recv_idx] + await ft.recv(left, tag=step)
        await send_work.wait()

    for step in range(world - 1):
        send_idx = (rank - step + 1) % world
        recv_idx = (rank - step) % world
        send_work = ft.isend(right, chunks[send_idx], tag=world + step)
        chunks[recv_idx] = await ft.recv(left, tag=world + step)
        await send_work.wait()

    return torch.cat(chunks)


expected = [torch.tensor([60, 64, 68, 72])] * 4
check(ring_all_reduce(flock.per_rank(values)), expected)